### Initial ML Model for Delsys Data Input (LDA Model)
Consists of:
- Filtering
- Windowing
- Feature Extraction
- Training LDA
- Evaluating accuracy
- Works for ADLs like “water-bottle lift” and “zipper”

First setting up virtual environment:
- py -m venv venv
- venv\Scripts\activate

Then setting up dependencies:
- pip install numpy scipy scikit-learn matplotlib seaborn

##### Imports

In [1]:
# Imports
import pandas as pd
import numpy as np
import os
import glob
from scipy.signal import butter, filtfilt, iirnotch
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

##### Functions

In [2]:
def load_emg_imu(csv_path):
    """
    Load EMG + IMU data from Delsys Trigno CSV with proper sensor identification.
    
    Returns:
        emg_data: dict {sensor_name: (n_samples, 1)}
        imu_data: dict {sensor_name: (n_samples, 6)} - [acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z]
        time_data: dict {sensor_name: (n_samples, 1)}
        fs_emg: EMG sampling frequency
        fs_imu: IMU sampling frequency
    """
    # Read sensor row (row 4)
    sensor_row = pd.read_csv(csv_path, header=None, skiprows=3, nrows=1, skipinitialspace=True)
    sensor_row = sensor_row.ffill(axis=1).iloc[0].tolist()

    # Read measurement row (row 6)
    meas_row = pd.read_csv(csv_path, header=None, skiprows=5, nrows=1, skipinitialspace=True)
    meas_row = meas_row.iloc[0].tolist()

    # Create unique column names
    combined_cols = []
    sensor_counter = {}
    for sensor, meas in zip(sensor_row, meas_row):
        sensor = str(sensor).strip() if pd.notna(sensor) else "Unknown"
        meas = str(meas).strip() if pd.notna(meas) else "Unknown"
        
        sensor_counter[sensor] = sensor_counter.get(sensor, 0) + 1
        unique_sensor = f"{sensor}_{sensor_counter[sensor]}"
        combined_cols.append(f"{unique_sensor}||{meas}")

    # Load data starting from row 9
    df = pd.read_csv(csv_path, header=None, skiprows=8, skipinitialspace=True, 
                     low_memory=False, on_bad_lines='skip')

    # Handle column count mismatch
    n_cols = df.shape[1]
    if len(combined_cols) < n_cols:
        for i in range(n_cols - len(combined_cols)):
            combined_cols.append(f"Extra_{i}||Unknown")
    elif len(combined_cols) > n_cols:
        combined_cols = combined_cols[:n_cols]

    df.columns = combined_cols

    # Convert to numeric
    df = df.apply(lambda x: pd.to_numeric(x.astype(str).str.strip(), errors='coerce'))
    df = df.dropna(how='all')  # Remove completely empty rows

    # Organize data by sensor
    sensor_data = {}
    
    for col in df.columns:
        if '||' not in col:
            continue
            
        sensor_name, meas_name = col.split('||')
        
        if sensor_name not in sensor_data:
            sensor_data[sensor_name] = {
                'emg': None,
                'time_emg': None,
                'acc_x': None, 'acc_y': None, 'acc_z': None,
                'gyro_x': None, 'gyro_y': None, 'gyro_z': None,
                'time_imu': None
            }
        
        data_col = df[col].dropna().values
        
        # Classify measurement type
        if '(mV)' in meas_name and 'EMG' in meas_name:
            sensor_data[sensor_name]['emg'] = data_col
        elif 'Time Series' in meas_name and 'EMG' in meas_name:
            sensor_data[sensor_name]['time_emg'] = data_col
        elif 'ACC X' in meas_name:
            sensor_data[sensor_name]['acc_x'] = data_col
        elif 'ACC Y' in meas_name:
            sensor_data[sensor_name]['acc_y'] = data_col
        elif 'ACC Z' in meas_name:
            sensor_data[sensor_name]['acc_z'] = data_col
        elif 'GYRO X' in meas_name:
            sensor_data[sensor_name]['gyro_x'] = data_col
        elif 'GYRO Y' in meas_name:
            sensor_data[sensor_name]['gyro_y'] = data_col
        elif 'GYRO Z' in meas_name:
            sensor_data[sensor_name]['gyro_z'] = data_col
        elif 'Time Series' in meas_name and 'ACC' in meas_name:
            sensor_data[sensor_name]['time_imu'] = data_col

    # Extract organized data
    emg_data = {}
    imu_data = {}
    time_data = {}
    
    for sensor_name, data in sensor_data.items():
        # EMG data
        if data['emg'] is not None and len(data['emg']) > 0:
            emg_data[sensor_name] = data['emg'].reshape(-1, 1)
            if data['time_emg'] is not None:
                time_data[sensor_name] = data['time_emg'].reshape(-1, 1)
        
        # IMU data (stack acc + gyro)
        imu_channels = []
        for key in ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']:
            if data[key] is not None and len(data[key]) > 0:
                imu_channels.append(data[key])
        
        if len(imu_channels) > 0:
            # Find minimum length to align all IMU channels
            min_len = min(len(ch) for ch in imu_channels)
            imu_channels = [ch[:min_len] for ch in imu_channels]
            imu_data[sensor_name] = np.column_stack(imu_channels)

    # Determine sampling frequencies from data
    fs_emg = 963  # Hz (from Delsys spec)
    fs_imu = 148.148  # Hz (from Delsys spec)
    
    return emg_data, imu_data, time_data, fs_emg, fs_imu

In [3]:
def extract_emg_features(window):
    """Extract comprehensive EMG features from a window"""
    # Time-domain features
    mav = np.mean(np.abs(window))
    rms = np.sqrt(np.mean(window**2))
    var = np.var(window)
    wl = np.sum(np.abs(np.diff(window)))
    
    # Zero crossings
    zc = np.sum(np.diff(np.sign(window)) != 0) / len(window)
    
    # Slope sign changes
    diff_signal = np.diff(window)
    ssc = np.sum(np.diff(np.sign(diff_signal)) != 0) / (len(window) - 1)
    
    # Frequency-domain features
    fft_vals = np.fft.rfft(window)
    power_spectrum = np.abs(fft_vals)**2
    freqs = np.fft.rfftfreq(len(window), 1/963)
    
    total_power = np.sum(power_spectrum)
    if total_power > 0:
        mnf = np.sum(freqs * power_spectrum) / total_power
        cumsum = np.cumsum(power_spectrum)
        mdf_idx = np.argmax(cumsum >= total_power/2)
        mdf = freqs[mdf_idx]
    else:
        mnf = 0
        mdf = 0
    
    return np.array([mav, rms, var, wl, zc, ssc, mnf, mdf])

In [4]:
def extract_imu_features(window):
    """Extract IMU features from a window (works for any number of axes)"""
    # Time-domain features
    mean_val = np.mean(window, axis=0)
    std_val = np.std(window, axis=0)
    rms_val = np.sqrt(np.mean(window**2, axis=0))
    range_val = np.max(window, axis=0) - np.min(window, axis=0)
    
    # Signal magnitude area (for 3-axis signals)
    if window.shape[1] >= 3:
        sma = np.mean(np.sum(np.abs(window[:, :3]), axis=1))
    else:
        sma = np.mean(np.sum(np.abs(window), axis=1))
    
    # Combine features
    features = np.concatenate([mean_val, std_val, rms_val, range_val, [sma]])
    return features

In [5]:
def window_and_extract_features(emg_dict, imu_dict, fs_emg=963, fs_imu=148.148,
                                 window_sec=0.20, overlap_sec=0.10):
    """
    Window EMG and IMU data separately (accounting for different sampling rates)
    and extract features from aligned windows.
    """
    # Calculate window parameters
    emg_win_size = int(window_sec * fs_emg)
    emg_step = int((window_sec - overlap_sec) * fs_emg)
    
    imu_win_size = int(window_sec * fs_imu)
    imu_step = int((window_sec - overlap_sec) * fs_imu)
    
    all_features = []
    
    # Sort sensors for consistent ordering
    emg_sensors = sorted(emg_dict.keys())
    imu_sensors = sorted(imu_dict.keys())
    
    # Find minimum number of windows across all sensors
    min_windows = float('inf')
    
    for sensor in emg_sensors:
        n_windows = (len(emg_dict[sensor]) - emg_win_size) // emg_step
        min_windows = min(min_windows, n_windows)
    
    for sensor in imu_sensors:
        n_windows = (len(imu_dict[sensor]) - imu_win_size) // imu_step
        min_windows = min(min_windows, n_windows)
    
    # Extract features window by window
    for win_idx in range(max(1, min_windows)):
        window_features = []
        
        # EMG features
        for sensor in emg_sensors:
            start = win_idx * emg_step
            end = start + emg_win_size
            
            if end > len(emg_dict[sensor]):
                break
                
            window = emg_dict[sensor][start:end].flatten()
            
            feats = extract_emg_features(window)
            window_features.extend(feats)
        
        # IMU features
        for sensor in imu_sensors:
            start = win_idx * imu_step
            end = start + imu_win_size
            
            if end > len(imu_dict[sensor]):
                break
                
            window = imu_dict[sensor][start:end]
            
            feats = extract_imu_features(window)
            window_features.extend(feats)
        
        if len(window_features) > 0:
            all_features.append(window_features)
    
    return np.array(all_features)

In [6]:
def process_trial(csv_path, label):
    """Load and process a single trial"""
    emg_dict, imu_dict, time_dict, fs_emg, fs_imu = load_emg_imu(csv_path)
    
    print(f"  Loaded: {len(emg_dict)} EMG sensors, {len(imu_dict)} IMU sensors")
    
    X = window_and_extract_features(emg_dict, imu_dict, fs_emg, fs_imu)
    y = np.full(X.shape[0], label)
    
    return X, y

In [7]:
def load_all_trials(class_trials):
    """Load all trials organized by class"""
    X_all = []
    y_all = []
    
    for label, files in sorted(class_trials.items()):
        print(f"\nProcessing Class {label}: {len(files)} files")
        for file in files:
            print(f"  File: {os.path.basename(file)}")
            try:
                X_trial, y_trial = process_trial(file, label)
                print(f"    Generated {X_trial.shape[0]} windows with {X_trial.shape[1]} features")
                X_all.append(X_trial)
                y_all.append(y_trial)
            except Exception as e:
                print(f"    ERROR: {e}")
                continue
    
    if len(X_all) == 0:
        raise ValueError("No data was successfully loaded!")
    
    X_all = np.vstack(X_all)
    y_all = np.concatenate(y_all)
    
    return X_all, y_all

In [8]:
def prepare_data(X, y, test_size=0.2, random_state=42):
    """Split and scale data"""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    return X_train, X_test, y_train, y_test, scaler

In [9]:
def train_evaluate(X_train, X_test, y_train, y_test, use_cv=True):
    """Train and evaluate LDA classifier"""
    clf = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
    
    if use_cv and len(np.unique(y_train)) > 1:
        cv_scores = cross_val_score(clf, X_train, y_train, cv=min(5, len(y_train)//2))
        print(f"\nCross-validation scores: {cv_scores}")
        print(f"Mean CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    
    clf.fit(X_train, y_train)
    
    y_pred = clf.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)
    
    print(f"\nTest Accuracy: {test_acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    return clf

##### Load and process ADL data

X_lift, y_lift = process_trial("20251113-Data/Lifting.csv", label=0)
X_zip, y_zip = process_trial("20251113-Data/Zipping.csv", label=1)
X_pinch, y_pinch = process_trial("20251113-Data/Pinching.csv", label=2)

X = np.vstack([X_lift, X_zip, X_pinch])
y = np.concatenate([y_lift, y_zip, y_pinch])

In [10]:
def get_trials_from_folder(data_dir):
    """
    Scan folder and parse filenames to assign class labels.
    Expects format: "0.1_name.csv", "1.2_name.csv", etc.
    """
    trial_files = glob.glob(os.path.join(data_dir, "*.csv"))
    class_trials = {}
    
    for f in trial_files:
        basename = os.path.basename(f)
        try:
            class_label = int(basename.split(".")[0])
        except (ValueError, IndexError):
            print(f"  Skipping file with unexpected format: {basename}")
            continue
        
        class_trials.setdefault(class_label, []).append(f)
    
    return class_trials

In [11]:
def get_trials_from_multiple_folders(data_dirs):
    """
    Scan multiple folders and combine all trials.
    
    Parameters:
        data_dirs: list of folder paths OR single folder path string
    
    Returns:
        class_trials: dict {class_label: [file_paths]}
        folder_info: dict with metadata about each folder
    """
    # Handle single folder or list of folders
    if isinstance(data_dirs, str):
        data_dirs = [data_dirs]
    
    combined_trials = {}
    folder_info = {}
    
    print(f"\nScanning {len(data_dirs)} folder(s) for data...")
    
    for folder in data_dirs:
        if not os.path.exists(folder):
            print(f"  ⚠ Warning: Folder not found: {folder}")
            continue
        
        print(f"\n  Folder: {folder}")
        folder_trials = get_trials_from_folder(folder)
        
        if len(folder_trials) == 0:
            print(f"    No valid CSV files found")
            continue
        
        # Track folder metadata
        folder_info[folder] = {
            'n_classes': len(folder_trials),
            'n_files': sum(len(files) for files in folder_trials.values()),
            'classes': list(folder_trials.keys())
        }
        
        # Merge into combined trials
        for label, files in folder_trials.items():
            combined_trials.setdefault(label, []).extend(files)
            print(f"    Class {label}: {len(files)} files")
    
    return combined_trials, folder_info

In [12]:
if __name__ == "__main__":
    data_dirs = [
        "20251201-Data",
        "20251202-Data"
    ]
    
    print("="*70)
    print("EMG + IMU CLASSIFICATION PIPELINE")
    print("="*70)
    
    # Parse files from all folders
    class_trials, folder_info = get_trials_from_multiple_folders(data_dirs)
    
    # Display summary
    print("\n" + "="*70)
    print("DATA SUMMARY")
    print("="*70)
    print(f"\nTotal folders processed: {len(folder_info)}")
    for folder, info in folder_info.items():
        print(f"\n  {folder}:")
        print(f"    Classes: {info['n_classes']}, Files: {info['n_files']}")
    
    print(f"\n{'='*70}")
    print(f"COMBINED DATA ACROSS ALL FOLDERS")
    print(f"{'='*70}")
    print(f"\nFound {len(class_trials)} unique classes:")
    for label, files in sorted(class_trials.items()):
        print(f"  Class {label}: {len(files)} files total")
    
    if len(class_trials) == 0:
        print(f"\n⚠ No valid CSV files found in any folder!")
        print("Please check:")
        print("  - Folder paths are correct")
        print("  - CSV files exist in folders")
        print("  - Filenames start with class number (e.g., '0.1_trial.csv')")
    else:
        # Load and process
        print("\n" + "="*70)
        print("LOADING AND FEATURE EXTRACTION")
        print("="*70)
        X, y = load_all_trials(class_trials)
        
        print(f"\n{'='*70}")
        print(f"DATASET STATISTICS")
        print(f"{'='*70}")
        print(f"Total samples: {X.shape[0]}")
        print(f"Features per sample: {X.shape[1]}")
        
        # Detailed class distribution
        class_counts = dict(zip(*np.unique(y, return_counts=True)))
        print(f"\nClass distribution:")
        for label in sorted(class_counts.keys()):
            count = class_counts[label]
            percentage = (count / len(y)) * 100
            print(f"  Class {label}: {count:5d} samples ({percentage:5.1f}%)")
        
        # Train and evaluate
        print(f"\n{'='*70}")
        print("TRAINING AND EVALUATION")
        print("="*70)
        X_train, X_test, y_train, y_test, scaler = prepare_data(X, y)
        print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")
        
        clf = train_evaluate(X_train, X_test, y_train, y_test, use_cv=True)

EMG + IMU CLASSIFICATION PIPELINE

Scanning 2 folder(s) for data...

  Folder: 20251201-Data
    Class 0: 6 files
    Class 1: 8 files
    Class 2: 8 files
    Class 3: 4 files

  Folder: 20251202-Data
    Class 0: 3 files
    Class 1: 4 files
    Class 2: 4 files
    Class 3: 2 files

DATA SUMMARY

Total folders processed: 2

  20251201-Data:
    Classes: 4, Files: 26

  20251202-Data:
    Classes: 4, Files: 13

COMBINED DATA ACROSS ALL FOLDERS

Found 4 unique classes:
  Class 0: 9 files total
  Class 1: 12 files total
  Class 2: 12 files total
  Class 3: 6 files total

LOADING AND FEATURE EXTRACTION

Processing Class 0: 9 files
  File: 0.1_hannah_20251201.csv
  Loaded: 12 EMG sensors, 72 IMU sensors
    Generated 317 windows with 456 features
  File: 0.1_meagan_20251201.csv
  Loaded: 12 EMG sensors, 72 IMU sensors
    Generated 352 windows with 456 features
  File: 0.2_hannah_20251201.csv
  Loaded: 12 EMG sensors, 72 IMU sensors
    Generated 315 windows with 456 features
  File: 0.2